# Libraries

Before we begin, as usual, we'll have to import the libraries that we need.

In [ ]:
import os
import sys
sys.path.append(os.path.abspath(os.pardir))

import pandas as pd
import mlflow
import argparse
from src import data_processing
from src import model_building

# Preparing Data and Model

Let's make sure that our functions are performing as expected without any issues by using them to load our data and our model. After making sure of its performance, we'll look to automate the whole training process by utilizing the train.py file to run using the command-line interface. We'll work on parsing the arguments from the command line to the python file by using the argparse module.

In [2]:
dataset_path = os.path.join('..','datasets','Philippine Fake News Corpus.csv')
df = pd.read_csv(dataset_path)
train, test = data_processing.process_data(
    df = df, 
    feature_col = 'Content',
    label_col = 'Label',
    random_state = 42,
    df_name = 'Philippine Fake News Corpus.csv'
)

## Verifying create_dataloaders Functionality

In [3]:
trainloader, testloader = model_building.create_dataloaders(train, test, 'Content', 'Label')

In [4]:
trainloader, testloader

(<torch.utils.data.dataloader.DataLoader at 0x2be53654980>,
 <torch.utils.data.dataloader.DataLoader at 0x2be53654950>)

Great, looks like our trainloader and testloaders were created successfully. Next, we'll have to verify our dataset by ensuring that it has the same shapes and datatypes.

## Verify Dataset

Right now, we're just sampling the dataloaders to get an overview of the data.

In [5]:
for batch in trainloader:
    sample_train = batch
    break

for batch in testloader:
    sample_test = batch
    break

print(f'x_train shape: {sample_train[0].shape} | x_test shape: {sample_test[0].shape}')
print(f'x_train dtype: {sample_train[0].dtype} | x_test shape: {sample_test[0].dtype}\n')

print(f'y_train shape: {sample_train[1].shape} | y_test shape: {sample_test[1].shape}')
print(f'y_train dtype: {sample_train[1].dtype} | y_test dtype: {sample_test[1].dtype}')

x_train shape: torch.Size([32, 24186]) | x_test shape: torch.Size([32, 24186])
x_train dtype: torch.int64 | x_test shape: torch.int64

y_train shape: torch.Size([32]) | y_test shape: torch.Size([32])
y_train dtype: torch.int64 | y_test dtype: torch.int64


Here, we also define the max sequence that is in the data, we'll use this later in creating our model. This variable could likely also change depending on the dataset that we have; however, we only currently have one dataset as implementing more datasets would require more tinkering with the functions we've already created, for now, we'll focus on automating the training process.

For this definition, we'll use the train and test sets instead of the dataloader as the dataloader will be created internally using the train_model function.

In [ ]:
max_seq = max(pd.concat([
    train['Content'],
    test['Content']
]).apply(len))
max_seq

24186

We expect that the samples should both have the same shapes and types. Here, we test that assumption by checking whether the train and test shapes match alongside with the datatypes.

In [7]:
expectations = {
    'Similar shapes': (
        (sample_train[0].shape == sample_test[0].shape) &
        (sample_train[1].shape == sample_test[1].shape)
    ),
    'Similar dtypes': (
        (sample_train[0].dtype == sample_test[0].dtype) &
        (sample_train[1].dtype == sample_test[1].dtype)
    )
}

expectations

{'Similar shapes': True, 'Similar dtypes': True}

# Build Model

With our dataloaders set, we now have to create our model. Usually, we'd create a config that we can change using the train.py functions, allowing us to create different models by only changing parameters within the function, but right now, let's take a closer look at what we need to create our model.

In [8]:
config = {
    'vocab_size': 8000,
    'embed_dim': 5,
    'pad_id': 3,
    'conv_dim': 4,
    'kernel_size': 5,
    'max_seq': max_seq,
}

model = model_building.FakeNewsDetector(**config)

Now that we've created our model, we can check whether this model matches our other models, since we want to increment the version accordingly if we're training an entirely different model than the ones we've already trained.

## Verifiying if its the same model as the logged models

To verify our model, we can check the parameters used for that model. Luckily, mlflow tracked these in the training loop. All we have to do now, is to query the mlflow database that we've set up to see the latest model. First, we have to define the model directory, which is the folder that we use to store all our models alongside their artifacts.

In [9]:
model_dir = os.path.join('..','models','FakeNewsDetector')
os.listdir(model_dir)

['225bddb08550475ba1bd78085bf05958',
 '4d2ed0343c554989a322540d8fc060bc',
 '859f1744b6ac44c8ad22f79ab61b658b',
 'e2f7a87641c748b99899751bbf1a8ce9',
 'f488585343114bbb87add6fd998d544a',
 'mlflow.db']

We can see that our model names are kind of gibberish, and this makes it really hard to read which of these models are the most recent. However, we are not really interested in manually looking at each folder. We are interested in the database file that mlflow used to store all the model runs: mlflow.db.

We'll get the experiment database record using the get_experiment function that we've defined under model_building.py. This lets us access all the tracked records of model training parameters, metrics, and other factors regarding training.

In [ ]:
model_building.get_experiment(
    exp_name = 'FakeNewsDetector',
    uri_path = os.path.join(model_dir, 'mlflow.db')
)

<Experiment: artifact_location=('file:///d:/zPersonal/Tools/VS Code/VSCode Script '
 'Folder/GithubRepoTemps/FakeNewsDetection/notebooks/../models/FakeNewsDetector'), creation_time=1779880707986, experiment_id='1', last_update_time=1779880707986, lifecycle_stage='active', name='FakeNewsDetector', tags={}, trace_location=None, workspace='default'>

Now that we've set the experiment to the correct database, we can query the database to see which model is the most recent. Additionally, we can sort by train and test loss to see the most recently optimal model.

In [ ]:
recent_runs = mlflow.search_runs(
    filter_string = "status = 'FINISHED'",
    order_by = ['end_time DESC', 'metrics.test_loss ASC', 'metrics.train_loss ASC'],
    search_all_experiments = True
)
recent_runs

,run_id,experiment_id,status,artifact_uri,start_time,end_time,metrics.train_loss,metrics.test_loss,metrics.Loss,params.max_seq,params.conv_dim,params.kernel_size,params.vocab_size,params.pad_id,params.embed_dim,tags.mlflow.runName,tags.mlflow.source.name,tags.mlflow.source.type,tags.version,tags.mlflow.user
0,e2f7a87641c748b99899751bbf1a8ce9,1,FINISHED,file:///d:/zPersonal/Tools/VS Code/VSCode Scri...,2026-06-02 09:21:53.360000+00:00,2026-06-02 09:43:58.695000+00:00,0.000015,0.002698,NaN,24186,4,5,8000,3,5,tasteful-fawn-292,model_building.ipynb,NOTEBOOK,1.0,kayle
1,4d2ed0343c554989a322540d8fc060bc,1,FINISHED,file:///d:/zPersonal/Tools/VS Code/VSCode Scri...,2026-06-01 10:01:19.649000+00:00,2026-06-01 10:29:53.726000+00:00,0.000010,0.003674,NaN,24186,4,5,8000,3,5,bold-moth-290,model_building.ipynb,NOTEBOOK,1.0,kayle
2,f488585343114bbb87add6fd998d544a,1,FINISHED,file:///d:/zPersonal/Tools/VS Code/VSCode Scri...,2026-05-30 09:41:40.357000+00:00,2026-05-30 09:48:27.933000+00:00,NaN,NaN,0.010168,24186,4,5,8000,3,5,charming-cat-820,model_building.ipynb,NOTEBOOK,1.0,kayle
3,859f1744b6ac44c8ad22f79ab61b658b,1,FINISHED,file:///d:/zPersonal/Tools/VS Code/VSCode Scri...,2026-05-30 08:46:25.649000+00:00,2026-05-30 08:53:31.161000+00:00,NaN,NaN,0.075763,24186,4,5,8000,3,5,sincere-cat-696,model_building.ipynb,NOTEBOOK,None,kayle
4,225bddb08550475ba1bd78085bf05958,1,FINISHED,file:///d:/zPersonal/Tools/VS Code/VSCode Scri...,2026-05-28 10:55:44.448000+00:00,2026-05-28 11:01:53.473000+00:00,NaN,NaN,2.548278,24186,4,5,8000,3,5,caring-bee-173,model_building.ipynb,NOTEBOOK,None,kayle


We can see in this dataframe that the run_id column matches the names of the folders that is within the same directory as our mlflow.db file. These are the models that we have trained. Thanks to our order_by parameter, we can see that the recently most optimal model is the one that starts with "e2f7a" with a train loss of 0.000015 and a test loss of 0.002698.

In addition to this, we can also see the parameters used in this model, which we will be using to check whether the model we create is different or not. Let's grab this specific row as the latest run by storing it in a variable

In [ ]:
latest_run = recent_runs.loc[0, :]
latest_run

run_id                                      e2f7a87641c748b99899751bbf1a8ce9
experiment_id                                                              1
status                                                              FINISHED
artifact_uri               file:///d:/zPersonal/Tools/VS Code/VSCode Scri...
start_time                                  2026-06-02 09:21:53.360000+00:00
end_time                                    2026-06-02 09:43:58.695000+00:00
metrics.train_loss                                                  0.000015
metrics.test_loss                                                   0.002698
metrics.Loss                                                             NaN
params.max_seq                                                         24186
params.conv_dim                                                            4
params.kernel_size                                                         5
params.vocab_size                                                       8000

In [ ]:
expected_params = {
    'vocab_size': int(latest_run['params.vocab_size']),
    'embed_dim': int(latest_run['params.embed_dim']),
    'pad_id': int(latest_run['params.pad_id']),
    'conv_dim': int(latest_run['params.conv_dim']),
    'kernel_size': int(latest_run['params.kernel_size']),
    'max_seq': int(latest_run['params.max_seq'])
}

for k,v in config.items():
    print(f'Same {k}: {v == expected_params[k]}')

Same vocab_size: True
Same embed_dim: True
Same pad_id: True
Same conv_dim: True
Same kernel_size: True
Same max_seq: True


## Load Latest Model 

In [14]:
uri_path = os.path.join(model_dir, 'model.db')

model_building.load_latest_model(model, model_dir)

# Setting Up Command Line Arguments

In [ ]:
parser = argparse.ArgumentParser(description = 'Train a Filipino Fake News Detector')
parser.add_argument('-e', '--epochs', type = int, help = 'Number of times the model sees the whole dataset in training')
parser.add_argument('-ed', '--embed_dim', type = int, default = 5, help = 'Number of embedding dimensions for the model')
parser.add_argument('-cd', '--conv_dim', type = int, default = 4, help = 'Number of convolutional dimensions for the model')
parser.add_argument('-k', '--kernel_size', type = int, default = 5, help = 'Kernel size of the Convolutional and Pooling Layers')
parser.add_argument('-vc', '--vocab_size', type = int, default = 8000, help = 'Size of the vocabulary the model is trained on')
parser.add_argument('-pid', '--pad_id', type = int, default = 3, help = 'Integer used to register as the padding token')
parser.add_argument('-b', '--batch_size', type = int, default = 32, help = 'Size of the batches for the dataloader the model uses')
parser.add_argument('-v', '--verbose', type = bool, default = True, help = 'Whether or not the program should output progress reports')
parser.add_argument('-rs', '--random_state', type = int, default = 42, help = 'Seed used for the psuedo-random functions in the program')

# Putting it together with command line args

## Prepare Data for Model

In [ ]:
dataset_dir = os.path.join('..','datasets')

csv_datasets = os.listdir(dataset_dir)
csv_datasets.remove('corpus.txt')

dataset_paths = [(csv, os.path.join(dataset_dir, csv)) for csv in csv_datasets]

print(dataset_paths)

In [ ]:
train, test = None, None

for df_name, dataset_path in dataset_paths:
    df = pd.read_csv(dataset_path)
    
    if train is None and test is None:
        train, test = data_processing.process_data(
            df = df,
            feature_col = 'Content',
            label_col = 'Label',
            random_state = args.random_state,
            df_name = df_name,
            vocab_size = args.vocab_size
        )
    else:
        train_subset, test_subset = data_processing.process_data(
            df = df,
            feature_col = 'Content',
            label_col = 'Label',
            random_state = args.random_state,
            df_name = df_name,
            vocab_size = args.vocab_size
        )
        
        train = pd.concat([
            train,
            train_subset
        ])
        
        test = pd.concat([
            test,
            test_subset
        ])

In [ ]:
display(train.head())
display(test.head())

In [ ]:
# Get max seq for model config
max_seq = max(pd.concat([
    train['Content'],
    test['Content']
]).apply(len))
max_seq

## Making the model

In [ ]:
config = {
    'vocab_size': args.vocab_size,
    'embed_dim': args.embed_dim,
    'pad_id': args.pad_id,
    'conv_dim': args.conv_dim,
    'kernel_size': args.kernel_size,
    'max_seq': max_seq,
}

In [ ]:
model = model_building.FakeNewsDetector(**config)

### Verifying Model Params with Expected Params

In [ ]:
# Check if the model has the same params as recently trained models, then increment model_version if true
model_dir = os.path.join('..','models','FakeNewsDetector')
os.listdir(model_dir)

model_building.get_experiment(
    exp_name = 'FakeNewsDetector',
    uri_path = os.path.join(model_dir, 'mlflow.db')
)

recent_runs = mlflow.search_runs(
    filter_string = "status = 'FINISHED'",
    order_by = ['end_time DESC', 'metrics.test_loss ASC', 'metrics.train_loss ASC'],
    search_all_experiments = True
)

latest_run = recent_runs.loc[0, :]

expected_params = {
    'vocab_size': int(latest_run['params.vocab_size']),
    'embed_dim': int(latest_run['params.embed_dim']),
    'pad_id': int(latest_run['params.pad_id']),
    'conv_dim': int(latest_run['params.conv_dim']),
    'kernel_size': int(latest_run['params.kernel_size']),
    'max_seq': int(latest_run['params.max_seq'])
}


param_mismatch = False
for k,v in config.items():
    print(f'Same {k}: {v == expected_params[k]}')
    
    if not v == expected_params[k]:
        param_mismatch = True
        
print('-'*24)
print(f'Same model used: {not param_mismatch}')

In [ ]:
# Increment model_version if parameters don't match to last trained model
model_version = float(latest_run['tags.version'])

if param_mismatch:
    model_version += 0.1
    
model_version

In [ ]:
model_building.train_model(
    model = model,
    model_config = config,
    epochs = args.epochs,
    datasets = (train, test),
    batch_size = args.batch_size
    verbose = args.verbose
    model_version = model_version
)

# Command Line Tool Tests

In [2]:
import argparse

In [3]:
parser = argparse.ArgumentParser()
parser.add_argument('-n', '--name', help = 'Your Name')

_StoreAction(option_strings=['-n', '--name'], dest='name', nargs=None, const=None, default=None, type=None, choices=None, required=False, help='Your Name', metavar=None)

In [4]:
args = parser.parse_args(args = ['-n Test'])

print(args.name)

 Test


In [25]:
parser = argparse.ArgumentParser(description = 'Train a Filipino Fake News Detector')
parser.add_argument('-e', '--epochs', type = int, help = 'Number of times the model sees the whole dataset in training')
parser.add_argument('-ed', '--embed_dim', type = int, default = 5, help = 'Number of embedding dimensions for the model')
parser.add_argument('-cd', '--conv_dim', type = int, default = 4, help = 'Number of convolutional dimensions for the model')
parser.add_argument('-k', '--kernel_size', type = int, default = 5, help = 'Kernel size of the Convolutional and Pooling Layers')
parser.add_argument('-vc', '--vocab_size', type = int, default = 8000, help = 'Size of the vocabulary the model is trained on')
parser.add_argument('-pid', '--pad_id', type = int, default = 3, help = 'Integer used to register as the padding token')
parser.add_argument('-b', '--batch_size', type = int, default = 32, help = 'Size of the batches for the dataloader the model uses')
parser.add_argument('-v', '--verbose', type = bool, default = True, help = 'Whether or not the program should output progress reports')
parser.add_argument('-rs', '--random_state', type = int, default = 42, help = 'Seed used for the psuedo-random functions in the program')

_StoreAction(option_strings=['-rs', '--random_state'], dest='random_state', nargs=None, const=None, default=42, type=<class 'int'>, choices=None, required=False, help='Seed used for the psuedo-random functions in the program', metavar=None)

In [26]:
args = parser.parse_args(args = ['--epochs=5'])
print(args)

Namespace(epochs=5, embed_dim=5, conv_dim=4, kernel_size=5, vocab_size=8000, pad_id=3, batch_size=32, verbose=True, random_state=42)


### Building the Dataset

In [11]:
dataset_dir = os.path.join('..','datasets')

csv_datasets = os.listdir(dataset_dir)
csv_datasets.remove('corpus.txt')

dataset_paths = [(csv, os.path.join(dataset_dir, csv)) for csv in csv_datasets]

print(dataset_paths)

[('Philippine Fake News Corpus.csv', '..\\datasets\\Philippine Fake News Corpus.csv')]


In [15]:
train, test = None, None

for df_name, dataset_path in dataset_paths:
    df = pd.read_csv(dataset_path)
    
    if train is None and test is None:
        train, test = data_processing.process_data(
            df = df,
            feature_col = 'Content',
            label_col = 'Label',
            random_state = args.random_state,
            df_name = df_name,
            vocab_size = args.vocab_size
        )
    else:
        train_subset, test_subset = data_processing.process_data(
            df = df,
            feature_col = 'Content',
            label_col = 'Label',
            random_state = args.random_state,
            df_name = df_name,
            vocab_size = args.vocab_size
        )
        
        train = pd.concat([
            train,
            train_subset
        ])
        
        test = pd.concat([
            test,
            test_subset
        ])

In [16]:
display(train.head())
display(test.head())

,Content,Label
0,"[397, 398, 7972, 5154, 8, 2589, 3706, 5252, 10...",1
1,"[542, 7972, 1242, 2722, 4, 1841, 2654, 6506, 7...",0
2,"[542, 7972, 196, 7075, 29, 1582, 7953, 7926, 5...",0
3,"[397, 398, 7972, 220, 4354, 137, 7964, 7934, 1...",1
4,"[397, 398, 7972, 220, 4126, 73, 6316, 790, 795...",1


,Content,Label
0,"[4020, 4110, 295, 7972, 2129, 1433, 1346, 7941...",1
1,"[1716, 1715, 7972, 716, 150, 1523, 73, 64, 566...",1
2,"[397, 398, 7972, 7419, 358, 76, 426, 6153, 65,...",1
3,"[542, 7972, 220, 1475, 133, 8, 774, 27, 1148, ...",0
4,"[397, 398, 7972, 7233, 926, 7941, 220, 397, 39...",1


In [23]:
# Finding max_seq
max_seq = max(pd.concat([
    train['Content'],
    test['Content']
]).apply(len))
max_seq

24186

### Building the Model

In [28]:
config = {
    'vocab_size': args.vocab_size,
    'embed_dim': args.embed_dim,
    'pad_id': args.pad_id,
    'conv_dim': args.conv_dim,
    'kernel_size': args.kernel_size,
    'max_seq': max_seq,
}

In [ ]:
model = model_building.FakeNewsDetector(**config)

### Training the model

In [36]:
# Check if the model has the same params as recently trained models, then increment model_version if true
model_dir = os.path.join('..','models','FakeNewsDetector')
os.listdir(model_dir)

model_building.get_experiment(
    exp_name = 'FakeNewsDetector',
    uri_path = os.path.join(model_dir, 'mlflow.db')
)

recent_runs = mlflow.search_runs(
    filter_string = "status = 'FINISHED'",
    order_by = ['end_time DESC', 'metrics.test_loss ASC', 'metrics.train_loss ASC'],
    search_all_experiments = True
)

latest_run = recent_runs.loc[0, :]

expected_params = {
    'vocab_size': int(latest_run['params.vocab_size']),
    'embed_dim': int(latest_run['params.embed_dim']),
    'pad_id': int(latest_run['params.pad_id']),
    'conv_dim': int(latest_run['params.conv_dim']),
    'kernel_size': int(latest_run['params.kernel_size']),
    'max_seq': int(latest_run['params.max_seq'])
}


param_mismatch = False
for k,v in config.items():
    print(f'Same {k}: {v == expected_params[k]}')
    
    if not v == expected_params[k]:
        param_mismatch = True
        
print('-'*24)
print(f'Same model used: {not param_mismatch}')

Same vocab_size: True
Same embed_dim: True
Same pad_id: True
Same conv_dim: True
Same kernel_size: True
Same max_seq: True
------------------------
Same model used: True


In [38]:
float(latest_run['tags.version'])


1.0

In [40]:
# Increment model_version if parameters don't match to last trained model
model_version = float(latest_run['tags.version'])

if param_mismatch:
    model_version += 0.1
    
model_version

1.0

In [ ]:
model_building.train_model(
    model = model,
    model_config = config,
    epochs = args.epochs,
    datasets = (train, test),
    batch_size = args.batch_size
    verbose = args.verbose
    model_version = model_version
)